In [2]:
%pip install -q hindsight-client groq gradio

import concurrent.futures
from google.colab import userdata
from hindsight_client import Hindsight
from groq import Groq

# Run Hindsight calls in a worker thread, so they never collide with Colab's event loop
_pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)
def in_thread(fn, *args, **kwargs):
    return _pool.submit(fn, *args, **kwargs).result()

hs = Hindsight(base_url="https://api.hindsight.vectorize.io",
               api_key=userdata.get("HINDSIGHT_API_KEY"))
groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
GROQ_MODEL = "openai/gpt-oss-120b"

def recall_similar(title, service, symptoms, top_k=8):
    query = f"{title}. Service: {service}. Symptoms: {symptoms}"
    resp = in_thread(hs.recall, bank_id="incident-memory", query=query)
    return resp.results[:top_k]

SYSTEM_PROMPT = """You are an Incident Response Agent for an engineering team.
You receive a live production incident and memories of similar past incidents.

Rules:
- Prefer fixes from past incidents that worked, and say which past incident you rely on.
- If no past incident is a good match, say so clearly and give a general investigation plan.
- Only cite a past incident using the memory text provided. Do not invent incident names or IDs.
- Be concrete and brief. Engineers are under pressure.

Format your answer with these sections:
1. Likely cause
2. Immediate steps (numbered)
3. Based on past incidents (quote the memory line you relied on, word for word)
4. How to verify the fix
5. Escalate if
"""

def triage(title, service, symptoms, severity="SEV2"):
    memories = recall_similar(title, service, symptoms)
    memory_text = "\n".join(f"- {m.text}" for m in memories) or "No similar past incidents found."
    user_msg = f"""LIVE INCIDENT
Title: {title}
Service: {service}
Severity: {severity}
Symptoms: {symptoms}

MEMORY (from Hindsight)
{memory_text}"""
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=1200,
    )
    return resp.choices[0].message.content

print("Setup ready.")

Setup ready.


In [9]:
import gradio as gr

def ui_triage(title, service, symptoms, severity):
    if not title or not symptoms:
        return "Please enter at least a title and symptoms."
    return triage(title, service, symptoms, severity)

with gr.Blocks(title="INFERON AI") as demo:
    gr.Markdown("# Incident Response Agent\nDescribe a live incident and get a plan based on past incidents.")
    with gr.Row():
        title = gr.Textbox(label="Incident title")
        service = gr.Textbox(label="Service")
    symptoms = gr.Textbox(label="Symptoms", lines=3)
    severity = gr.Dropdown(["SEV1", "SEV2", "SEV3"], value="SEV2", label="Severity")
    btn = gr.Button("Get response plan")
    output = gr.Markdown()
    btn.click(ui_triage, [title, service, symptoms, severity], output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c8f923ba521d808b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
seed_incidents = [
    dict(title="Checkout API 503 errors after deploy",
         service="checkout-api",
         symptoms="HTTP 503 spikes, pods restarting, memory usage climbing to limit after release v2.4",
         root_cause="Memory leak in the new payment-retry cache that never evicted entries",
         resolution_steps="1. Roll back to v2.3 via deploy pipeline. 2. Confirm pods are healthy. 3. Patch cache eviction and redeploy behind a canary",
         runbook="RB-012 Rollback procedure",
         severity="SEV1"),
    dict(title="Database connection pool exhausted",
         service="orders-db",
         symptoms="Timeouts on order lookups, 'too many connections' in logs, latency up 10x",
         root_cause="A batch job opened connections without closing them, starving the pool",
         resolution_steps="1. Kill the batch job. 2. Restart the connection pooler. 3. Add connection timeout and close handler to the batch job",
         runbook="RB-031 DB connection recovery",
         severity="SEV2"),
    dict(title="CDN cache serving stale assets",
         service="frontend-cdn",
         symptoms="Users see old JavaScript bundle, login page broken after release, some regions affected",
         root_cause="Cache TTL too long and build hash not included in asset filenames",
         resolution_steps="1. Purge CDN cache for /static/*. 2. Verify new bundle hash in headers. 3. Enable hashed filenames in build config",
         runbook="RB-044 CDN purge",
         severity="SEV2"),
    dict(title="Kafka consumer lag on payments topic",
         service="payments-consumer",
         symptoms="Consumer lag growing into millions, payment confirmations delayed by 20 minutes",
         root_cause="A poison message caused the consumer to crash-loop on deserialization",
         resolution_steps="1. Identify offset of poison message. 2. Skip it and send to dead-letter topic. 3. Add schema validation before deserialization",
         runbook="RB-027 Consumer lag recovery",
         severity="SEV1"),
]

def incident_text(inc):
    return (f"Incident: {inc['title']}. Service: {inc['service']}. Severity: {inc['severity']}. "
            f"Symptoms: {inc['symptoms']}. Root cause: {inc['root_cause']}. "
            f"Resolution steps: {inc['resolution_steps']}. Runbook: {inc['runbook']}.")

# Seed data is already stored in Hindsight. Do not run the loop below again,
# or the incidents will be stored twice.
# for inc in seed_incidents:
#     in_thread(hs.retain, bank_id="incident-memory", content=incident_text(inc))

In [5]:
def record_resolution(title, service, symptoms, root_cause, resolution_steps, runbook,
                      severity, worked, notes=""):
    outcome = "worked" if worked else "did NOT work"
    content = (f"Incident: {title}. Service: {service}. Severity: {severity}. "
               f"Symptoms: {symptoms}. Root cause: {root_cause}. "
               f"Resolution steps: {resolution_steps}. Runbook: {runbook}. "
               f"Outcome: the fix {outcome}. {notes}")
    in_thread(hs.retain, bank_id="incident-memory", content=content)
    print(f"Saved resolved incident '{title}' (fix {outcome}).")

# Only run this once per resolved incident. Running it again stores a duplicate.
# record_resolution(
#     title="Payments API 503 after v3.1 release",
#     service="payments-api",
#     symptoms="503 errors, pods restarting, memory climbing after release v3.1",
#     root_cause="Same retry cache memory leak pattern as checkout-api",
#     resolution_steps="1. Roll back to v3.0. 2. Confirm pod health. 3. Patch cache eviction",
#     runbook="RB-012 Rollback procedure",
#     severity="SEV1",
#     worked=True,
#     notes="Rollback restored service in 6 minutes.",
# )

In [8]:
resp = hs.recall(bank_id="incident-memory",
                 query="payments-api 503 v3.1 rollback")
print("Total results:", len(resp.results))
for m in resp.results:
    print("-", m.text)
    print()

Total results: 102
- Service restored by rolling back to v3.0 and patching cache eviction, following RB-012 rollback procedure. | When: 2026-11-08 | Where: payments-api | Rollback restored service in 6 minutes.

- Service restored after rolling back to v3.0 and patching cache eviction. | When: 2026-11-08 | Where: payments-api | Rollback procedure RB-012 completed in 6 minutes.

- The rollback restored service in 6 minutes.

- Service restored after rolling back to v3.0 and patching cache eviction. | When: 2026-11-08 | Where: payments-api | Rollback procedure RB-012 resolved the incident in 6 minutes.

- Service was restored by rolling back to v3.0 and patching cache eviction following RB-012 rollback procedure.

- Service restored in 6 minutes by rolling back to v3.0 and patching cache eviction. | When: 2026-11-08 | Where: payments-api | Resolution of SEV1 incident.

- Rollback procedure RB-012 completed in 6 minutes.

- Payments API experienced SEV1 incident with 503 errors, pod resta

In [6]:
import json, re

POSTMORTEM_PROMPT = """Read this incident postmortem and return ONLY a JSON object with two keys:
"lessons" (a short string: what we learned about the cause and detection)
"prevention" (a short string: concrete actions to stop it happening again)

Postmortem:
{raw_text}"""

def parse_json(text):
    """Pull a JSON object out of a model reply, even if wrapped in ```json fences."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group(0))

def add_postmortem(title, service, raw_text):
    reply = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": POSTMORTEM_PROMPT.format(raw_text=raw_text)}],
        max_tokens=600,
    ).choices[0].message.content
    data = parse_json(reply)

    content = (f"Postmortem for {title} ({service}). "
               f"Lessons: {data['lessons']} "
               f"Prevention: {data['prevention']}")
    in_thread(hs.retain, bank_id="incident-memory", content=content)
    print("Postmortem saved to memory.")
    return data

# Run add_postmortem(...) only when you have a new postmortem to save.
# Running it again stores a duplicate.

In [8]:
%pip install -q gradio

import gradio as gr

def ui_triage(title, service, symptoms, severity):
    if not title or not symptoms:
        return "Please enter at least a title and symptoms."
    return triage(title, service, symptoms, severity)

with gr.Blocks(title="INFERON AI") as demo:
    gr.Markdown("# INFERON AI\nDescribe a live incident and get a plan based on past incidents.")
    with gr.Row():
        title = gr.Textbox(label="Incident title")
        service = gr.Textbox(label="Service")
    symptoms = gr.Textbox(label="Symptoms", lines=3)
    severity = gr.Dropdown(["SEV1", "SEV2", "SEV3"], value="SEV2", label="Severity")
    btn = gr.Button("Get response plan")
    output = gr.Markdown()
    btn.click(ui_triage, [title, service, symptoms, severity], output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://872d6f9bf71b0f429f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
